In [14]:
from pymavlink import mavutil
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Load log ──────────────────────────────────────────────────────────────────
mlog = mavutil.mavlink_connection('/Users/ethansimon/Desktop/Flight_Logs/00000005.BIN')

target_msgs = ['IMU', 'ATT', 'BAT', 'MOT', 'RATE', 'RCOU']
data = {msg: [] for msg in target_msgs}

while True:
    msg = mlog.recv_match(type=target_msgs, blocking=False)
    if msg is None:
        break
    data[msg.get_type()].append(msg.to_dict())

dfs = {k: pd.DataFrame(v) for k, v in data.items() if data[k]}
print("Loaded:", list(dfs.keys()))


# ── Helper ────────────────────────────────────────────────────────────────────
def make_fig(title, rows, row_titles, height=700):
    fig = make_subplots(rows=rows, cols=1,
                        shared_xaxes=True,
                        subplot_titles=row_titles,
                        vertical_spacing=0.06)
    fig.update_layout(title=title, height=height, template='plotly_dark')
    return fig


# ── 1. IMU Vibration ──────────────────────────────────────────────────────────
if 'IMU' in dfs:
    imu = dfs['IMU']
    fig = make_fig('IMU Vibration', 3, ['AccX', 'AccY', 'AccZ'])
    for i, axis in enumerate(['AccX', 'AccY', 'AccZ'], start=1):
        if axis in imu.columns:
            fig.add_trace(go.Scatter(x=imu['TimeUS'], y=imu[axis],
                                     mode='lines', name=axis), row=i, col=1)
            fig.add_hline(y=15,  line_dash='dash', line_color='red', row=i, col=1)
            fig.add_hline(y=-15, line_dash='dash', line_color='red', row=i, col=1)
    fig.show()


# ── 2. Attitude Tracking ──────────────────────────────────────────────────────
if 'ATT' in dfs:
    att = dfs['ATT']
    axes_pairs = [('DesRoll', 'Roll'), ('DesPitch', 'Pitch'), ('DesYaw', 'Yaw')]
    fig = make_fig('Attitude Tracking', 3, ['Roll', 'Pitch', 'Yaw'])
    for i, (desired, actual) in enumerate(axes_pairs, start=1):
        if desired in att.columns and actual in att.columns:
            fig.add_trace(go.Scatter(x=att['TimeUS'], y=att[desired],
                                     mode='lines', name=f'Desired {actual}',
                                     line=dict(color='orange')), row=i, col=1)
            fig.add_trace(go.Scatter(x=att['TimeUS'], y=att[actual],
                                     mode='lines', name=f'Actual {actual}',
                                     line=dict(color='cyan')), row=i, col=1)
    fig.show()


# ── 3. Battery ────────────────────────────────────────────────────────────────
if 'BAT' in dfs:
    bat = dfs['BAT']
    fig = make_fig('Battery Performance', 2, ['Voltage (V)', 'Current (A)'])
    if 'Volt' in bat.columns:
        fig.add_trace(go.Scatter(x=bat['TimeUS'], y=bat['Volt'],
                                 mode='lines', name='Voltage',
                                 line=dict(color='yellow')), row=1, col=1)
    if 'Curr' in bat.columns:
        fig.add_trace(go.Scatter(x=bat['TimeUS'], y=bat['Curr'],
                                 mode='lines', name='Current',
                                 line=dict(color='orange')), row=2, col=1)
        # Overlay power (W) on current plot if both available
        if 'Volt' in bat.columns:
            power = bat['Volt'] * bat['Curr']
            fig.add_trace(go.Scatter(x=bat['TimeUS'], y=power,
                                     mode='lines', name='Power (W)',
                                     line=dict(color='red', dash='dot')), row=2, col=1)
    fig.show()


# ── 4. Motor Balance ──────────────────────────────────────────────────────────
# Try MOT first, fall back to RCOU
motor_source = 'MOT' if 'MOT' in dfs else 'RCOU' if 'RCOU' in dfs else None

if motor_source:
    mot = dfs[motor_source]
    # Find motor columns — varies by firmware
    motor_cols = [c for c in mot.columns
                  if c.startswith('Mot') or c.startswith('C')
                  and c[1:].isdigit()]
    if not motor_cols:
        # RCOU uses C1, C2 ... style
        motor_cols = [c for c in mot.columns if c in
                      ['C1','C2','C3','C4','C5','C6']]

    if motor_cols:
        colors = ['cyan','orange','lime','red','violet','yellow']
        fig = make_fig(f'Motor Outputs ({motor_source}) — Balance Check',
                       1, ['Motor Output %'])
        for j, col in enumerate(motor_cols[:6]):
            fig.add_trace(go.Scatter(x=mot['TimeUS'], y=mot[col],
                                     mode='lines', name=col,
                                     line=dict(color=colors[j % len(colors)])),
                          row=1, col=1)
        fig.show()


# ── 5. Rate Controller (PID effort) ───────────────────────────────────────────
if 'RATE' in dfs:
    rate = dfs['RATE']
    axes_pairs = [('RDes', 'R'), ('PDes', 'P'), ('YDes', 'Y')]
    labels = ['Roll Rate', 'Pitch Rate', 'Yaw Rate']
    fig = make_fig('Rate Controller — Desired vs Actual', 3, labels)
    for i, (desired, actual) in enumerate(axes_pairs, start=1):
        if desired in rate.columns and actual in rate.columns:
            fig.add_trace(go.Scatter(x=rate['TimeUS'], y=rate[desired],
                                     mode='lines', name=f'Des {labels[i-1]}',
                                     line=dict(color='orange')), row=i, col=1)
            fig.add_trace(go.Scatter(x=rate['TimeUS'], y=rate[actual],
                                     mode='lines', name=f'Act {labels[i-1]}',
                                     line=dict(color='cyan')), row=i, col=1)
    fig.show()

Loaded: ['IMU', 'ATT', 'BAT', 'RATE', 'RCOU']
